# Q1: Ranking Clients Before a Call

Run the cells in order in Google Colab. Upload `bank-additional-full.csv` when prompted.

**Question:** Can we rank clients in a later period to capture more subscribers than random selection within the same contact budget?

The workflow selects a model using chronological validation, then evaluates the fixed configuration on chronological and random test sets. No classification threshold is needed for ranking.

## Step 1 — Import libraries
Colab includes the required libraries. This notebook uses the same model settings as the original analysis; library versions may differ.

In [1]:
import io
import json
import math
import platform
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from google.colab import files
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

SEED = 42
OUTPUT_DIR = Path('/content/q1_results')
OUTPUT_DIR.mkdir(exist_ok=True)
print({'python': platform.python_version(), 'sklearn': sklearn.__version__,
       'pandas': pd.__version__, 'numpy': np.__version__})

{'python': '3.13.18', 'sklearn': '1.6.1', 'pandas': '2.2.3', 'numpy': '2.4.6'}


## Step 2 — Upload and inspect the dataset
Choose the full semicolon-separated CSV, not the smaller sample. Keep the original row order.

In [2]:
uploaded = files.upload()
csv_names = [name for name in uploaded if name.lower().endswith('.csv')]
if len(csv_names) != 1:
    raise ValueError('Upload exactly one CSV: bank-additional-full.csv.')

df = pd.read_csv(io.BytesIO(uploaded[csv_names[0]]), sep=';')
assert len(df) == 41188, 'Expected 41,188 rows from bank-additional-full.csv.'
assert df.shape[1] == 21, 'Expected 21 columns. Check the delimiter and dataset version.'
assert set(df['y'].unique()) == {'yes', 'no'}
print('Dataset shape:', df.shape)
display(df.head())
display(df['y'].value_counts().rename('count').to_frame())

Saving bank-additional-full.csv to bank-additional-full (1).csv
Dataset shape: (41188, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


,count
y,
no,36548
yes,4640


## Step 3 — Define pre-call inputs
Exclude the target `y` and the unavailable call duration. We also exclude `campaign` conservatively because it includes the current campaign contact count. Keep the remaining 18 inputs, including previous campaign outcome. Treat their availability before dialing as an input assumption.

In [3]:
X = df.drop(columns=['y', 'duration', 'campaign']).copy()
y = df['y'].eq('yes').astype(int).to_numpy()
row_ids = np.arange(len(df))
categorical_columns = X.select_dtypes(include='object').columns.tolist()
numerical_columns = X.select_dtypes(include='number').columns.tolist()
print('Number of input features:', X.shape[1])
print('Categorical features:', categorical_columns)
print('Numerical features:', numerical_columns)

Number of input features: 18
Categorical features: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']
Numerical features: ['age', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']


## Step 4 — Create the chronological split
[UCI](https://archive.ics.uci.edu/dataset/222/bank+marketing) states that the full dataset is ordered by date. Use the earliest 60% for training, the next 20% for validation, and the final 20% for testing. The first 80% is the development set. Do not shuffle.

In [4]:
train_ids = row_ids[:24712]
validation_ids = row_ids[24712:32950]
development_ids = row_ids[:32950]
time_test_ids = row_ids[32950:]

assert not set(development_ids).intersection(time_test_ids)
split_sizes = pd.DataFrame([
    {'set': 'Training', 'first_row': 1, 'last_row': 24712, 'n': len(train_ids)},
    {'set': 'Validation', 'first_row': 24713, 'last_row': 32950, 'n': len(validation_ids)},
    {'set': 'Time test', 'first_row': 32951, 'last_row': len(df), 'n': len(time_test_ids)},
])
display(split_sizes)

,set,first_row,last_row,n
0,Training,1,24712,24712
1,Validation,24713,32950,8238
2,Time test,32951,41188,8238


## Step 5 — Define ranking metrics
Select the top 10% of the set being evaluated, rounding up. Break score ties by original row number.

- **Precision@10%:** subscribers among selected clients / selected clients.
- **Recall@10%:** selected subscribers / all subscribers in that set.
- **Lift@10%:** precision / overall subscription rate in the same set. Random selection has expected lift 1.

The 10% contact budget is fixed before testing.

In [5]:
def ranking_metrics(ids, scores):
    ids = np.asarray(ids)
    scores = np.asarray(scores)
    labels = y[ids]
    order = np.lexsort((ids, -scores))
    contact_count = math.ceil(0.10 * len(ids))
    captured = int(labels[order[:contact_count]].sum())
    subscribers = int(labels.sum())
    prevalence = float(labels.mean())
    precision = captured / contact_count
    return {
        'n': len(ids),
        'subscribers': subscribers,
        'subscription_rate': prevalence,
        'contacted': contact_count,
        'captured': captured,
        'precision_at_10': precision,
        'recall_at_10': captured / subscribers if subscribers else np.nan,
        'lift_at_10': precision / prevalence if prevalence else np.nan,
        'random_expected_captured': contact_count * prevalence,
    }

## Step 6 — Define candidate models
Compare logistic regression with two random forest configurations. Every pipeline fits its own preprocessing on training data only. Unknown categories are handled by the encoder. No resampling or class weights are used.

In [6]:
MODEL_NAMES = [
    'Logistic regression',
    'Random forest depth 5',
    'Random forest depth 10',
]

def make_model(name):
    preprocessing = ColumnTransformer([
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_columns),
        ('numerical', StandardScaler(), numerical_columns),
    ])
    if name == 'Logistic regression':
        estimator = LogisticRegression(C=1, max_iter=2000, random_state=SEED)
    else:
        depth = int(name.split()[-1])
        estimator = RandomForestClassifier(
            n_estimators=200,
            max_depth=depth,
            min_samples_leaf=20,
            random_state=SEED,
            n_jobs=2,
        )
    return Pipeline([('preprocessing', preprocessing), ('model', estimator)])

## Step 7 — Select a model using validation data
Choose the highest validation Lift@10%. In a tie, keep the earlier candidate in the list, which favors the simpler configuration. Do not use test results for selection.

In [7]:
validation_rows = []
fitted_candidates = {}
for name in MODEL_NAMES:
    pipeline = make_model(name)
    pipeline.fit(X.iloc[train_ids], y[train_ids])
    scores = pipeline.predict_proba(X.iloc[validation_ids])[:, 1]
    validation_rows.append({'model': name, **ranking_metrics(validation_ids, scores)})
    fitted_candidates[name] = pipeline

validation_results = pd.DataFrame(validation_rows)
selected_name = max(validation_rows, key=lambda row: row['lift_at_10'])['model']
display(validation_results)
print('Selected model:', selected_name)

,model,n,subscribers,subscription_rate,contacted,captured,precision_at_10,recall_at_10,lift_at_10,random_expected_captured
0,Logistic regression,8238,912,0.110706,824,42,0.050971,0.046053,0.460415,91.222141
1,Random forest depth 5,8238,912,0.110706,824,111,0.134709,0.121711,1.216810,91.222141
2,Random forest depth 10,8238,912,0.110706,824,102,0.123786,0.111842,1.118150,91.222141


Selected model: Random forest depth 5


## Step 8 — Check validation stability over time
Keep the selected training-fitted model fixed. Evaluate it on the earlier and later halves of the validation period. This is a development diagnostic, not an independent test.

In [8]:
selected_training_model = fitted_candidates[selected_name]
stability_rows = []
for label, ids in [
    ('Earlier validation half', validation_ids[:4119]),
    ('Later validation half', validation_ids[4119:]),
]:
    scores = selected_training_model.predict_proba(X.iloc[ids])[:, 1]
    stability_rows.append({'period': label, **ranking_metrics(ids, scores)})

stability_results = pd.DataFrame(stability_rows)
display(stability_results)

,period,n,subscribers,subscription_rate,contacted,captured,precision_at_10,recall_at_10,lift_at_10,random_expected_captured
0,Earlier validation half,4119,418,0.101481,412,45,0.109223,0.107656,1.076294,41.810148
1,Later validation half,4119,494,0.119932,412,69,0.167476,0.139676,1.396422,49.411993


## Step 9 — Create the random comparison
Keep the EDA split: stratified 80% development and 20% test, with seed 42. Use the configuration already selected through chronological validation. These experiments share the same source data; they are not independent external validations.

In [9]:
random_development_ids, random_test_ids = train_test_split(
    row_ids, test_size=0.2, random_state=SEED, stratify=y
)
assert not set(random_development_ids).intersection(random_test_ids)
print('Random development size:', len(random_development_ids))
print('Random test size:', len(random_test_ids))

Random development size: 32950
Random test size: 8238


## Step 10 — Refit and run final evaluation
All model choices are now fixed. Refit separately on each development set, then evaluate its test set. Report the results without tuning against test scores.

In [10]:
test_rows = []
prediction_frames = []
for split_name, fit_ids, test_ids in [
    ('Time-based', development_ids, time_test_ids),
    ('Random', random_development_ids, random_test_ids),
]:
    final_model = make_model(selected_name)
    final_model.fit(X.iloc[fit_ids], y[fit_ids])
    scores = final_model.predict_proba(X.iloc[test_ids])[:, 1]
    test_rows.append({
        'split': split_name, 'model': selected_name,
        **ranking_metrics(test_ids, scores),
    })
    prediction_frames.append(pd.DataFrame({
        'split': split_name,
        'original_row': test_ids + 1,
        'actual_yes': y[test_ids],
        'score': scores,
    }))

test_results = pd.DataFrame(test_rows)
test_predictions = pd.concat(prediction_frames, ignore_index=True)
display(test_results)

,split,model,n,subscribers,subscription_rate,contacted,captured,precision_at_10,recall_at_10,lift_at_10,random_expected_captured
0,Time-based,Random forest depth 5,8238,2540,0.308327,824,487,0.591019,0.191732,1.916857,254.061665
1,Random,Random forest depth 5,8238,928,0.112649,824,414,0.502427,0.446121,4.460124,92.822530


## Step 11 — Interpret Q1
Compare each model result with random selection in the same test set. The chronological test is the main result. Different subscription rates can produce different lift values across splits. Without client IDs, this evaluates later-period records rather than proving performance on entirely unseen people. Earlier EDA already accessed the full dataset.

In [11]:
for result in test_rows:
    print(f"{result['split']} test")
    print(f"  Selected model: {selected_name}")
    print(f"  Overall subscription rate: {result['subscription_rate']:.2%}")
    print(f"  Clients contacted: {result['contacted']:,}")
    print(f"  Actual subscribers captured: {result['captured']:,}")
    print(f"  Random expected capture: {result['random_expected_captured']:.2f}")
    print(f"  Precision@10%: {result['precision_at_10']:.2%}")
    print(f"  Recall@10%: {result['recall_at_10']:.2%}")
    print(f"  Lift@10%: {result['lift_at_10']:.3f}")
    print()

Time-based test
  Selected model: Random forest depth 5
  Overall subscription rate: 30.83%
  Clients contacted: 824
  Actual subscribers captured: 487
  Random expected capture: 254.06
  Precision@10%: 59.10%
  Recall@10%: 19.17%
  Lift@10%: 1.917

Random test
  Selected model: Random forest depth 5
  Overall subscription rate: 11.26%
  Clients contacted: 824
  Actual subscribers captured: 414
  Random expected capture: 92.82
  Precision@10%: 50.24%
  Recall@10%: 44.61%
  Lift@10%: 4.460



## Step 12 — Export results
Download the tables, row-level predictions, split membership, and runtime versions as a ZIP file.

In [12]:
validation_results.to_csv(OUTPUT_DIR / 'validation_results.csv', index=False)
stability_results.to_csv(OUTPUT_DIR / 'stability_results.csv', index=False)
test_results.to_csv(OUTPUT_DIR / 'test_results.csv', index=False)
test_predictions.to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)

manifest = pd.DataFrame({
    'original_row': row_ids + 1,
    'time_split': np.where(row_ids < 24712, 'training',
                           np.where(row_ids < 32950, 'validation', 'test')),
    'random_split': 'development',
})
manifest.loc[random_test_ids, 'random_split'] = 'test'
manifest.to_csv(OUTPUT_DIR / 'split_manifest.csv', index=False)
metadata = {
    'selected_model': selected_name,
    'excluded_columns': ['y', 'duration', 'campaign'],
    'seed': SEED,
    'contact_fraction': 0.10,
    'versions': {'python': platform.python_version(), 'sklearn': sklearn.__version__,
                 'pandas': pd.__version__, 'numpy': np.__version__},
}
(OUTPUT_DIR / 'metadata.json').write_text(json.dumps(metadata, indent=2))
archive = Path('/content/q1_results.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for file in sorted(OUTPUT_DIR.iterdir()):
        if file.is_file():
            bundle.write(file, arcname=file.name)
files.download(str(archive))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>